In [1]:
import json
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from dotenv import load_dotenv
import os
load_dotenv()
BASE_URL = "https://dev.to/api"
DEV_API_KEY = os.getenv("DEV_API_KEY")

# Set a user agent to avoid being blocked as a bot by the API edge.
DEFAULT_HEADERS = {
    "Accept": "application/json",
    "User-Agent": "CAP5771-DEV-Client/1.0 (contact: student@example.com)",
    "api-key": DEV_API_KEY,

}

def fetch_articles(page=1, per_page=10, tag=None, username=None, headers=None):
    params = {
        "page": page,
        "per_page": per_page,
    }
    if tag:
        params["tag"] = tag
    if username:
        params["username"] = username

    url = f"{BASE_URL}/articles?{urlencode(params)}"
    request_headers = dict(DEFAULT_HEADERS)
    if headers:
        request_headers.update(headers)

    request = Request(url, headers=request_headers)

    try:
        with urlopen(request) as response:
            payload = response.read().decode("utf-8")
            return json.loads(payload)
    except HTTPError as exc:
        detail = exc.read().decode("utf-8")
        raise RuntimeError(f"DEV API error {exc.code}: {detail}") from exc

articles = fetch_articles(page=1, per_page=5, tag="python")

for article in articles:
    print(f"{article['title']} — {article['url']}")


Checking Django Settings — https://dev.to/adamghill/checking-django-settings-12g4
pytest-aitest: Unit Tests Can't Test Your MCP Server. AI Can. — https://dev.to/sbroenne/pytest-aitest-unit-tests-cant-test-your-mcp-server-ai-can-1ebn
Two Ways to Move Tensors Without Stopping: Inside vLLM's Async GPU Transfer Patterns — https://dev.to/mketkar/two-ways-to-move-tensors-without-stopping-inside-vllms-async-gpu-transfer-patterns-dk7
Day 5 of Python(Tuple) — https://dev.to/vidya_varshini/day-5-of-pythontuple-4ga2
🔑Beginner-Friendly Guide 'Longest Balanced Substring II' - Problem 3714 (C++, Python, JavaScript) — https://dev.to/om_shree_0709/beginner-friendly-guide-longest-balanced-substring-ii-problem-3714-c-python-javascript-591f


In [2]:
import json
import re
from urllib.request import Request, urlopen
from urllib.error import HTTPError
import time


def fetch_article_by_id(article_id, headers=None, retries=3):
    url = f"{BASE_URL}/articles/{article_id}"
    request_headers = dict(DEFAULT_HEADERS)
    if headers:
        request_headers.update(headers)

    for attempt in range(retries):
        try:
            request = Request(url, headers=request_headers)
            print(f"[FETCH] Attempt {attempt + 1}/{retries} for article {article_id}")
            with urlopen(request) as response:
                payload = response.read().decode("utf-8")
                return json.loads(payload)
        except HTTPError as exc:
            if exc.code == 404:
                print(f"[FETCH] Article {article_id} not found (404), skipping")
                return None
            if exc.code == 429:
                print(f"[FETCH] Rate limited (429), waiting 5s before retry...")
                time.sleep(5)
                continue
            detail = exc.read().decode("utf-8")
            print(f"[FETCH] HTTP error {exc.code}: {detail}")
            if attempt == retries - 1:
                return None
            time.sleep(1)
        except Exception as e:
            print(f"[FETCH] Error on attempt {attempt + 1}: {e}")
            if attempt == retries - 1:
                return None
            time.sleep(1)
    
    return None


def markdown_to_text(markdown):
    # Remove code blocks and inline code
    markdown = re.sub(r"```[\s\S]*?```", " ", markdown)
    markdown = re.sub(r"`[^`]*`", " ", markdown)

    # Remove images and links but keep link text
    markdown = re.sub(r"!\[[^\]]*\]\([^\)]*\)", " ", markdown)
    markdown = re.sub(r"\[([^\]]+)\]\([^\)]*\)", r"\1", markdown)

    # Remove HTML tags
    markdown = re.sub(r"<[^>]+>", " ", markdown)

    # Strip markdown formatting characters
    markdown = re.sub(r"[#>*_~\-]+", " ", markdown)

    # Normalize whitespace
    return " ".join(markdown.split())

In [5]:
import sqlite3
import time
from datetime import datetime, timedelta, timezone

DB_PATH = "dev_ai_articles.sqlite"
TAGS = ["ai"]
PER_PAGE = 30
DAYS_BACK = 20


def parse_iso8601(value):
    if not value:
        return None
    if value.endswith("Z"):
        value = value[:-1] + "+00:00"
    try:
        return datetime.fromisoformat(value)
    except ValueError:
        return None


def ensure_schema(connection):
    print("[DB] Creating schema if needed...")
    connection.execute(
        """
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            url TEXT NOT NULL,
            published_at TEXT,
            tags TEXT,
            body_text TEXT,
            impressions INTEGER
        )
        """
    )
    connection.commit()
    print("[DB] Schema ready")


def upsert_article(connection, article):
    connection.execute(
        """
        INSERT INTO articles (id, title, url, published_at, tags, body_text, impressions)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(id) DO UPDATE SET
            title=excluded.title,
            url=excluded.url,
            published_at=excluded.published_at,
            tags=excluded.tags,
            body_text=excluded.body_text,
            impressions=excluded.impressions
        """,
        (
            article["id"],
            article["title"],
            article["url"],
            article.get("published_at"),
            article.get("tags"),
            article.get("body_text"),
            article.get("impressions"),
        ),
    )


def fetch_tagged_articles(tag, cutoff):
    page = 1
    article_count = 0
    while True:
        print(f"[API] Fetching page {page} for tag '{tag}'...")
        try:
            batch = fetch_articles(page=page, per_page=PER_PAGE, tag=tag)
        except Exception as e:
            print(f"[ERROR] Failed to fetch page {page}: {e}")
            break
            
        if not batch:
            print(f"[API] No more articles for tag '{tag}'")
            break

        print(f"[API] Got {len(batch)} articles on page {page}")
        for item in batch:
            published = parse_iso8601(item.get("published_at"))
            if published and published < cutoff:
                print(f"[API] Reached cutoff date, stopping tag '{tag}'")
                return
            yield item
            article_count += 1

        page += 1
        print(f"[API] Sleeping 0.4s before next page...")
        time.sleep(0.4)
    
    print(f"[API] Total articles yielded for tag '{tag}': {article_count}")


cutoff_date = datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)
print(f"[MAIN] Cutoff date: {cutoff_date}")

connection = sqlite3.connect(DB_PATH)
ensure_schema(connection)

seen_ids = set()
total_fetched = 0
total_saved = 0

for tag in TAGS:
    print(f"\n[MAIN] Starting tag: {tag}")
    for item in fetch_tagged_articles(tag, cutoff_date):
        article_id = item["id"]
        if article_id in seen_ids:
            print(f"[DEDUP] Skipping duplicate article {article_id}")
            continue
        seen_ids.add(article_id)
        total_fetched += 1
        print(f"[FETCH] Fetching full body for article {article_id} ({total_fetched} so far)...")
        
        try:
            detail = fetch_article_by_id(article_id)
        except Exception as e:
            print(f"[ERROR] Failed to fetch article {article_id}: {e}")
            continue
        
        if detail is None:
            print(f"[SKIP] Skipping article {article_id} due to fetch failure")
            continue
            
        body_markdown = detail.get("body_markdown", "")
        body_text = markdown_to_text(body_markdown)
        

        record = {
            "id": article_id,
            "title": detail.get("title", ""),
            "url": detail.get("url", ""),
            "published_at": detail.get("published_at"),
            "tags": ",".join(detail.get("tag_list", [])),
            "body_text": body_text,
            "impressions": detail.get("impressions"),
        }

        upsert_article(connection, record)
        total_saved += 1
        print(f"[DB] Saved article {article_id} (total: {total_saved})")

connection.commit()
connection.close()

print(f"\n[MAIN] Complete. Fetched: {total_fetched}, Saved: {total_saved}")

[MAIN] Cutoff date: 2026-01-24 16:55:15.396626+00:00
[DB] Creating schema if needed...
[DB] Schema ready

[MAIN] Starting tag: ai
[API] Fetching page 1 for tag 'ai'...
[API] Got 30 articles on page 1
[FETCH] Fetching full body for article 3254071 (1 so far)...
[FETCH] Attempt 1/3 for article 3254071
[DB] Saved article 3254071 (total: 1)
[FETCH] Fetching full body for article 3252542 (2 so far)...
[FETCH] Attempt 1/3 for article 3252542
[DB] Saved article 3252542 (total: 2)
[FETCH] Fetching full body for article 3253666 (3 so far)...
[FETCH] Attempt 1/3 for article 3253666
[DB] Saved article 3253666 (total: 3)
[FETCH] Fetching full body for article 3254672 (4 so far)...
[FETCH] Attempt 1/3 for article 3254672
[DB] Saved article 3254672 (total: 4)
[FETCH] Fetching full body for article 3254639 (5 so far)...
[FETCH] Attempt 1/3 for article 3254639
[DB] Saved article 3254639 (total: 5)
[FETCH] Fetching full body for article 3251337 (6 so far)...
[FETCH] Attempt 1/3 for article 3251337
[DB]

KeyboardInterrupt: 

In [2]:
import sqlite3

DB_PATH = "dev_ai_articles.sqlite"

with sqlite3.connect(DB_PATH) as conn:
    count = conn.execute("SELECT COUNT(*) FROM articles").fetchone()[0]
    print(f"Articles in DB: {count}")

Articles in DB: 85
